In [ ]:
import * as tslab from "tslab";
import { readFileSync } from "fs";

const css = readFileSync("../style.css", "utf-8");
tslab.display.html(`<style>${css}</style>`);

# Evaluation of Formulas from First Order Logic

In this notebook we show how formulas from *first order logic* can be evaluated in Python.

## The Axioms of Group Theory

To have a nontrivial example of formulas, we use the formulas from 
[group theory](https://en.wikipedia.org/wiki/Group_theory).  
A [group](https://en.wikipedia.org/wiki/Group_(mathematics)) is defined as a triple 
$$ \langle G, \mathrm{e}, \circ \rangle $$
where 
- $G$ is a non-empty set,
- $\mathrm{e}$ is an element from $G$, and
- $\circ:G \times G \rightarrow G$ is a binary function on $G$.
- Furthermore, the following axioms have to be satisfied:
  * $\forall x: \mathrm{e} \circ x = x$,
  * $\forall x: \exists{y}: y \circ x = \mathrm{e}$,
  * $\forall x: \forall y: \forall z: (x \circ y) \circ z = x \circ (y \circ z)$.
- A group is <em style="color:blue">commutative</em> if, additionally, the following formula is satisfied:
  $$\forall x: \forall y: x \circ y = y \circ x. $$

The notebook `FOL-Parser.ipynb` contains a notebook implementing a parser for first order logic.

In [ ]:
import { LogicParser } from './FOL-Parser';

We import a parser for FOL formulas.  This parser distinguishes between variables and function symbol as follows:
- A word starting with a lower case letter is interpreted as a *variable*.
- A word starting with an upper case letter is assumed to be a *function* or 
  *predicate symbol*.

Therefore, we represent the symbols from group theory as follows:
- The neutral element $\mathrm{e}$ of group theory is represented as the nullary function symbol `E`.
- As our parser does not support using the symbol $\circ$ as a binary operator, we will use the function symbol     
  `Multiply` to represent this operator.
- The predicate symbol $=$ is repesented as `Equals`

Then the formulas of group theory can be represented as follows:

In [ ]:
const G1 = '∀x:Equals(Multiply(E(),x),x)';

In [ ]:
const G2 = '∀x:∃y:Equals(Multiply(y,x),E())';

In [ ]:
const G3 = '∀x:∀y:∀z:Equals(Multiply(Multiply(x,y),z), Multiply(x,Multiply(y,z)))';

In [ ]:
const G4 = '∀x:∀y:Equals(Multiply(x,y), Multiply(y,x))';

The function $\texttt{parse}(s)$ takes a string $s$ and converts it into a nested tuple.

In [ ]:
type Formula = string | [string, ...Formula[]];

In [ ]:
function parse(s: string): Formula {
    const p = new LogicParser(s);
    return p.parse();
}

In [ ]:
const F1 = parse(G1);
console.dir(F1,{depth: null});

In [ ]:
const F2 = parse(G2);
console.dir(F2,{depth: null});

In [ ]:
const F3 = parse(G3);
console.dir(F3,{depth: null});

In [ ]:
const F4 = parse(G4);
console.dir(F4,{depth: null});

## A Structure for Group Theory

The smallest non-trivial group has just two elements.  Therefore, we can define the universe `U` as follows:

In [ ]:
const U: Set<number> = new Set([0, 1]);

Next, we need to define the nullary function that represents the nullary function `E`.  We define this function as a dictionary mapping the empty tuple into the element `0`. 

In [ ]:
const NeutralElement: { [key: string]: number } = { '()': 0 };

The binary function symbol `Multiply` is implemented as the dictionary `Product`:

In [ ]:
const Product: { [key: string]: number } = {
  '(0, 0)': 0,
  '(0, 1)': 1,
  '(1, 0)': 1,
  '(1, 1)': 0
};

The predicate symbol `Equals` is implemented as the binary relation `Identity`.

In [ ]:
const Identity: Set<string> = new Set(
  Array.from(U).map((x) => `(${x}, ${x})`)
);

Identity;

Now the interpretation $\mathcal{J}$ can be implemented as a dictionary mapping symbols to dictionaries that interpret these symbols. 

In [ ]:
const J: { [key: string]: any } = {
  E: NeutralElement,
  Multiply: Product,
  Equals: Identity
};

Next, we define the *first order structure* $\mathcal{S}$ as the pair $(\mathcal{U}, \mathcal{J})$.

In [ ]:
const S: [Set<number>, { [key: string]: any }] = [U, J];
S;

Finally, we define the *variable assignment* $\mathcal{I}$ for the variables $x$, $y$, and $z$. 

In [ ]:
const I: { [key: string]: number } = { x: 0, y: 1, z: 0 };
I;

## Functions to Evaluate Formulas

In TypeScript, if we use the spread syntax `...` in a destructuring assignment, then it can consume an arbitrary number of elements.
In the code below, the variable `R` collects all elements from the list `L` with the exception of the first element, which is assigned to `x`.

In [ ]:
const L: number[] = [1, 2, 3, 4];
const [x, ...R] = L;

console.log(x, R);

The procedure $\texttt{evalTerm}(t, \mathcal{S}, \mathcal{I})$ evaluates the term $t$ in the structure $\mathcal{S}$ using the variable assignment $\mathcal{I}$.

In [ ]:
function evalTerm(
  t: Formula,
  S: [Set<any>, { [key: string]: any }],
  I: { [key: string]: number }
): number {
  if (typeof t === "string") {
    if (!(t in I)) {
       throw new Error(`Variable '${t}' is not defined in the assignment I.`);
    }
    return I[t];
  }
  const [, J] = S; // J is the dictionary of interpretations
  const [f, ...Args] = t; // function symbol and array of arguments
  const fJ = J[f]; // interpretation of function symbol
  const ArgValsTuple = Args.map(arg => evalTerm(arg, S, I)); 
  const argKey =
    Args.length === 0
      ? "()"
      : "(" + ArgValsTuple.join(", ") + ")";
  if (!(argKey in fJ)) {
      throw new Error(`Function '${f}' is not defined for arguments ${argKey}.`);
  }
  return fJ[argKey];
}

In [ ]:
const t = parse('Multiply(E(),x)');
t;

In [ ]:
evalTerm(t, S, I);

This procedure evaluates the atomic formula a in the structure S using the variable assignment I.

In [ ]:
function evalAtomic(
  a: Formula,
  S: [Set<any>, { [key: string]: any }],
  I: { [key: string]: number }
): boolean {
  if (!Array.isArray(a)) {
    throw new Error(`evalAtomic expects a predicate call (tuple), but got a variable: "${a}"`);
  }
  const [, J] = S;
  const [p, ...Args] = a as [string, ...Formula[]];
  const pJ = J[p];
  const ArgValsTuple = Args.map((arg) => evalTerm(arg, S, I));
  const argKey =
    Args.length === 0
      ? "()"
      : "(" + ArgValsTuple.join(", ") + ")";
  return pJ.has(argKey);
}

In [ ]:
const f = parse('Equals(Multiply(E(),x),x)');
f;

In [ ]:
evalAtomic(f, S, I);

Given a variable assignment $\mathcal{I}$, a variable $x$, and an element $c$ from the universe $\mathcal{U}$, the function $\texttt{modify}(\mathcal{I}, x, c)$ computes the variable assignment $\mathcal{I}[x/c]$ which is defined for all variables $y$ as follows:
$$ 
I[x/c](y) = \left\{ \begin{array}{ll}
                        c     & \mbox{if $x = y$,}  \\
                        I(y)  & \mbox{otherwise.}
                        \end{array}
               \right.
$$

In [ ]:
function modify(
  I: { [key: string]: number },
  x: string,
  c: number
): { [key: string]: number } {
  const J = { ...I };
  J[x] = c;
  return J;
}

Given a first order logic formula $F$, a structure $\mathcal{S}$, and a variable assignment $\mathcal{I}$, the function $\texttt{evalFormula}(F, \mathcal{S}, \mathcal{I})$ computes the truth value of the formula $F$.

In [ ]:
function evalFormula(F: Formula, S: [Set<any>, { [key: string]: any }], I: { [key: string]: number }): boolean {
  const [U, _] = S;
  if (Array.isArray(F) && F.length === 1) {
    if (F[0] === '⊤') return true;
    if (F[0] === '⊥') return false;
  }
  if (Array.isArray(F)) {
      const op = F[0];
      if (op === '¬') {
          return !evalFormula(F[1], S, I);
      }
      if (op === '∧') {
          return evalFormula(F[1], S, I) && evalFormula(F[2], S, I);
      }
      if (op === '∨') {
          return evalFormula(F[1], S, I) || evalFormula(F[2], S, I);
      }
      if (op === '→') {
          return evalFormula(F[1], S, I) || evalFormula(F[2], S, I);
      }
      if (op === '↔') {
          return evalFormula(F[1], S, I) === evalFormula(F[2], S, I);
      }
      if (op === '∀') {
          const [, x, G] = F as [string, string, Formula];
          for (const c of U) {
              if (!evalFormula(G, S, modify(I, x, c))) {
                  return false;
              }
          }
          return true;
      }
      if (op === '∃') {
          const [, x, G] = F as [string, string, Formula];
          for (const c of U) {
              if (evalFormula(G, S, modify(I, x, c))) {
                  return true;
              }
          }
          return false;
      }
  }
  return evalAtomic(F, S, I);
}

## Checking whether $\mathcal{S}$ is a Group

In [ ]:
console.log(`evalFormula(${G1}, S, I) = ${evalFormula(F1, S, I)}`);
console.log(`evalFormula(${G2}, S, I) = ${evalFormula(F2, S, I)}`);
console.log(`evalFormula(${G3}, S, I) = ${evalFormula(F3, S, I)}`);
console.log(`evalFormula(${G4}, S, I) = ${evalFormula(F4, S, I)}`);

This shows that the structure $\mathcal{S}$ defined above is indeed a group.  Furthermore, it is a *commutative* group.

## Another Example

Let's show that the formula $\forall x: \exists y:p(x,y) \rightarrow \exists y:\forall x:p(x,y)$ is not *universally valid*, i.e. let's show the following:
$$ \not\models \forall x: \exists y:p(x,y) \rightarrow \exists y:\forall x:p(x,y) $$

In [ ]:
const G = '∀x:∃y:P(x,y)→∃y:∀x:P(x,y)';

In [ ]:
const F = parse(G);
console.dir(F,{depth: null});;

Our aim is to construct a structure $\mathcal{S} = \langle\mathcal{U}, \mathcal{J} \rangle$  such that 
$$
\mathcal{S}(F) = \mathtt{false}.
$$ 

In [ ]:
const U: Set<number> = new Set([0, 1]);

In [ ]:
const pJ: Set<string> = new Set(['(0, 0)', '(1, 1)']);

In [ ]:
const J: { [key: string]: Set<string> } = { P: pJ };

In [ ]:
const S: [Set<number>, { [key: string]: Set<string> }] = [U, J];

In [ ]:
const I: { [key: string]: number } = { x: 0, y: 0 };

In [ ]:
evalFormula(F, S, I);